<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Deep_learning_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cell 1 — GPU check + setup

In [1]:
!nvidia-smi
import os
from pathlib import Path

ROOT = Path("/content/av_perception")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

Fri Feb  6 21:02:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Cell 2 — Install YOLOv8 + utils

In [2]:
!pip -q install ultralytics opencv-python imageio imageio-ffmpeg

from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Cell 3 — Download dataset (YOLO-ready, no login)

In [3]:
from ultralytics.utils.downloads import download
from pathlib import Path
import shutil # Import shutil for rmtree

DATASETS_DIR = Path("datasets")
DATASETS_DIR.mkdir(exist_ok=True)

url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"

# Remove existing coco128 directory and zip file to ensure a clean re-download
coco128_dir = DATASETS_DIR / "coco128"
coco128_zip = DATASETS_DIR / "coco128.zip"

if coco128_dir.exists():
    shutil.rmtree(coco128_dir)
if coco128_zip.exists():
    coco128_zip.unlink()

download(url, dir=DATASETS_DIR)

Unzipping datasets/coco128.zip to /content/av_perception/datasets/coco128...: 100% ━━━━━━━━━━━━ 263/263 2.9Kfiles/s 0.1s


In [4]:
!ls datasets
!ls datasets/coco128

coco128  coco128.zip
images	labels	LICENSE  README.txt


In [5]:
from pathlib import Path
import yaml

yaml_data = {
    "path": "datasets/coco128",
    "train": "images/train2017",
    "val": "images/train2017",   # ✅ SAME as train
    "names": [
        "person",
        "bicycle",
        "car",
        "motorcycle",
        "airplane",
        "bus",
        "train",
        "truck"
    ]
}

yaml_path = Path("datasets/coco128/coco128.yaml")
with open(yaml_path, "w") as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False)

print("YAML fixed for single-split dataset")

YAML fixed for single-split dataset


In [6]:
!ls datasets

coco128  coco128.zip


In [7]:
!ls datasets/coco128

coco128.yaml  images  labels  LICENSE  README.txt


In [8]:
!ls datasets/coco128/images

train2017


In [9]:
!ls datasets/coco128/labels

train2017


### Cell 4 — Inspect dataset

In [10]:
import os, yaml

print(os.listdir("datasets"))
print(os.listdir("datasets/coco128"))

with open("datasets/coco128/coco128.yaml") as f:
    data_yaml = yaml.safe_load(f)

data_yaml

['coco128', 'coco128.zip']
['LICENSE', 'images', 'README.txt', 'labels', 'coco128.yaml']


{'path': 'datasets/coco128',
 'train': 'images/train2017',
 'val': 'images/train2017',
 'names': ['person',
  'bicycle',
  'car',
  'motorcycle',
  'airplane',
  'bus',
  'train',
  'truck']}

### Cell 5 — Train YOLOv8 (T4-friendly)

In [11]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="datasets/coco128/coco128.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    name="av_coco128",
)

Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/coco128/coco128.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=av_coco128, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspectiv

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a6508116780>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

### Cell 6 — Validate

In [12]:
model = YOLO("runs/detect/road_detection/weights/best.pt")
model.val(data="road_data/data.yaml")

FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/road_detection/weights/best.pt'

### Cell 7 — Download demo driving video

In [ ]:
!wget -q -O traffic.mp4 https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4

### Cell 8 — Detection + tracking + TTC overlay (AV logic)

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture("traffic.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    "traffic_ttc.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H),
)

FOCAL = 700
REAL_HEIGHT = {"person": 1.7, "car": 1.5}
track_hist = {}
TTC_THRESH = 2.0

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t = frame_id / fps
    results = model.track(frame, persist=True, conf=0.25, verbose=False)[0]

    if results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids   = results.boxes.id.cpu().numpy()
        clss  = results.boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, ids, clss):
            x1,y1,x2,y2 = map(int, box)
            label = model.names[cls]

            h = max(1, y2 - y1)
            dist = (REAL_HEIGHT.get(label,1.6)*FOCAL)/h

            ttc = None
            if tid in track_hist:
                d_prev, t_prev = track_hist[tid]
                v_rel = (d_prev - dist)/(t - t_prev + 1e-3)
                if v_rel > 0:
                    ttc = dist/v_rel

            track_hist[tid] = (dist,t)

            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,255),2)
            txt = f"{label} id={int(tid)} d~{dist:.1f}m"
            if ttc and ttc < 10:
                txt += f" TTC~{ttc:.1f}s"
            cv2.putText(frame,txt,(x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)

            if ttc and ttc < TTC_THRESH:
                cv2.putText(frame,"NEAR MISS!",
                            (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX,1.1,(0,0,255),3)

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
print("Saved traffic_ttc.mp4")

### Cell 9 — Convert to GIF

In [ ]:
import imageio

reader = imageio.get_reader("traffic_ttc.mp4")
frames = [reader.get_data(i) for i in range(0,120,3)]
imageio.mimsave("traffic_ttc.gif", frames, fps=8)

print("Saved traffic_ttc.gif")

### Cell 10 — Display

In [ ]:
from IPython.display import Video, Image

Video("traffic_ttc.mp4", embed=True)
Image("traffic_ttc.gif")